# Deposits

In [ ]:
#| default_exp deposit

In [ ]:
#| export

from dataclasses import dataclass
from typing import Optional
from sugar.pool import LiquidityPool

In [ ]:
#| export

# NOTE: for CL pools, `pool.type` is the tick spacing (Sugar's convention).

@dataclass(frozen=True)
class DepositQuote:
    """Returned by `chain.quote_*_deposit` and accepted by `chain.deposit`.

    CL fields (`tick_lower`, `tick_upper`, `sqrt_price_x96`) are required when
    `pool.is_cl`, must stay at defaults otherwise. `sqrt_price_x96` is set by
    the SDK from `initial_price` — consumers don't compute it themselves.
    """
    pool: LiquidityPool
    amount_token0: int
    amount_token1: int
    # CL-only fields
    tick_lower: Optional[int] = None
    tick_upper: Optional[int] = None
    # Q64.96 sqrt(P); used by NFPM only when initializing an uninitialized pool. SDK-computed.
    sqrt_price_x96: int = 0

    def __post_init__(self):
        if self.pool.is_cl:
            if self.tick_lower is None or self.tick_upper is None:
                raise ValueError("CL DepositQuote requires tick_lower and tick_upper")
            if self.tick_lower >= self.tick_upper:
                raise ValueError("tick_lower must be < tick_upper")
            ts = self.pool.type
            if self.tick_lower % ts != 0 or self.tick_upper % ts != 0:
                raise ValueError(f"tick bounds must be multiples of tick spacing ({ts})")
        else:
            if self.tick_lower is not None or self.tick_upper is not None or self.sqrt_price_x96 != 0:
                raise ValueError("basic DepositQuote must not set tick / sqrt_price_x96 fields")

In [ ]:
# Construction + guards. The on-chain deposit happy path is not covered by automated tests,
# matching the existing pattern for `swap`. Run manually on supersim.
from types import SimpleNamespace
from fastcore.test import test_eq, test_fail

basic_pool = SimpleNamespace(is_cl=False)
cl_pool = SimpleNamespace(is_cl=True, type=200,
                          token0=SimpleNamespace(decimals=18),
                          token1=SimpleNamespace(decimals=6))

# basic quote: tick / sqrt fields stay at defaults
q = DepositQuote(pool=basic_pool, amount_token0=1_000, amount_token1=2_000)
test_eq(q.tick_lower, None)
test_eq(q.sqrt_price_x96, 0)

# basic quote rejects stray CL fields
test_fail(lambda: DepositQuote(pool=basic_pool, amount_token0=1, amount_token1=1, tick_lower=0, tick_upper=200))
test_fail(lambda: DepositQuote(pool=basic_pool, amount_token0=1, amount_token1=1, sqrt_price_x96=1))

# CL quote: ticks required, ordered, and aligned to tick spacing
q = DepositQuote(pool=cl_pool, amount_token0=10**18, amount_token1=2*10**9, tick_lower=-200, tick_upper=200)
test_eq(q.tick_lower < q.tick_upper, True)
test_fail(lambda: DepositQuote(pool=cl_pool, amount_token0=1, amount_token1=1))                       # missing ticks
test_fail(lambda: DepositQuote(pool=cl_pool, amount_token0=1, amount_token1=1, tick_lower=10, tick_upper=10))  # not ordered
test_fail(lambda: DepositQuote(pool=cl_pool, amount_token0=1, amount_token1=1, tick_lower=1, tick_upper=2))    # misaligned

In [ ]:
#| hide

import nbdev; nbdev.nbdev_export()